# Module 4 Lab 3 — CNN Experiments

**Module 4: CNNs, Pretrained Models & Deep Dream | Week 1 | Lab (Session 4.6)**

---

You built a CNN for CIFAR-10 in Lab 2. Now let's experiment with the architecture and look inside the model to see what it learned.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Setup (same as Lab 2)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_data = datasets.CIFAR10('data', train=True, download=True, transform=transform)
test_data = datasets.CIFAR10('data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Reusable functions
def train_model(model, train_loader, test_loader, device, epochs=10, lr=0.001):
    """Train a model and return training history."""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = {'loss': [], 'acc': []}

    for epoch in range(epochs):
        model.train()
        total_loss, n = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n += 1
        history['loss'].append(total_loss / n)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                correct += (model(images).argmax(1) == labels).sum().item()
                total += labels.size(0)
        history['acc'].append(correct / total)
        print(f"  Epoch {epoch+1:2d}  loss={history['loss'][-1]:.4f}  acc={history['acc'][-1]:.2%}")

    return history

---

## Experiment 1: Filter Size

### Task 1: Compare 3×3 vs 5×5 filters

Build two CNNs with the same structure but different filter sizes:
- Model A: all `kernel_size=3`
- Model B: all `kernel_size=5`

Use 2 conv layers (32 and 64 filters), each followed by ReLU + MaxPool2d(2), then FC. Adjust padding so spatial dimensions work out.

Train both for 10 epochs and compare accuracy and parameter counts.

In [ ]:
# Task 1: Your code here


---

## Experiment 2: Number of Filters

### Task 2: Compare few vs many filters

Build three CNNs with 3×3 filters but different filter counts:
- Small: 8, 16 filters
- Medium: 32, 64 filters
- Large: 64, 128 filters

Train each for 10 epochs. Plot accuracy curves on the same graph. Print parameter counts.

In [ ]:
# Task 2: Your code here


---

## Experiment 3: Depth

### Task 3: Compare 2 vs 3 vs 4 conv layers

Build three CNNs with increasing depth (all using 3×3 filters, padding=1):
- 2 layers: 32 → 64
- 3 layers: 32 → 64 → 64
- 4 layers: 32 → 64 → 64 → 128

Each conv layer followed by ReLU, pool every 2 layers. Train for 10 epochs each.

In [ ]:
# Task 3: Your code here


---

## Looking Inside: Visualize Learned Filters

### Task 4: Display first-layer filters

Take your best model from the experiments above. Extract the first conv layer's weights and display them as images.

The first layer has shape `(out_channels, 3, 3, 3)` — each filter is a 3×3 RGB pattern. Display them in a grid.

Hint: normalize each filter to [0, 1] for display.

In [ ]:
# Task 4: Your code here


---

### Task 5: Visualize feature maps

Pass a test image through your CNN and capture the output after each conv layer (before pooling). Display the feature maps from the first conv layer as a grid.

Hint: Use forward hooks or break the forward pass into stages.

In [ ]:
# Task 5: Your code here


What do the feature maps look like? You should see some that highlight edges, some that respond to colors, and some that look like noise (not every filter learns something interpretable).

---

### Task 6: Results summary

Fill in a summary table with your results:

| Experiment | Config | Parameters | Test Accuracy |
|:-----------|:-------|:-----------|:--------------|
| Filter size | 3×3 | | |
| Filter size | 5×5 | | |
| Num filters | Small (8,16) | | |
| Num filters | Medium (32,64) | | |
| Num filters | Large (64,128) | | |
| Depth | 2 conv layers | | |
| Depth | 3 conv layers | | |
| Depth | 4 conv layers | | |

Which single change had the biggest impact on accuracy?

In [ ]:
# Task 6: Print your results table


---

## Challenge: Batch normalization

`nn.BatchNorm2d` normalizes activations between layers and often improves training. Add `BatchNorm2d` after each conv layer (before ReLU) in your best architecture. Does it help? How much?

In [ ]:
# Challenge: Your code here
